## Imports

In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host= 'processing-2.processing.data.production.internal',
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/geo-consistency'
print(local_dataset)

/Users/E2074/analytics/customer/segmentation/geo-consistency
/Users/E2074/local-datasets/customer/geo-consistency


## Datasets & Parameter

In [5]:
city = 'Bangalore'
date_str = '20250316'
date_obj = datetime.strptime(date_str, "%Y%m%d")

start_date = date_obj - timedelta(days=90)
end_date = date_obj

date_list = [(start_date + timedelta(days=i)).strftime("%Y%m%d") for i in range(91)]

print("start_date =", start_date.strftime("%Y%m%d"))
print("end_date =", end_date.strftime("%Y%m%d"))
print("date_range =", len(date_list))
print("list_date =", date_list)

start_date = 20241216
end_date = 20250316
date_range = 91
list_date = ['20241216', '20241217', '20241218', '20241219', '20241220', '20241221', '20241222', '20241223', '20241224', '20241225', '20241226', '20241227', '20241228', '20241229', '20241230', '20241231', '20250101', '20250102', '20250103', '20250104', '20250105', '20250106', '20250107', '20250108', '20250109', '20250110', '20250111', '20250112', '20250113', '20250114', '20250115', '20250116', '20250117', '20250118', '20250119', '20250120', '20250121', '20250122', '20250123', '20250124', '20250125', '20250126', '20250127', '20250128', '20250129', '20250130', '20250131', '20250201', '20250202', '20250203', '20250204', '20250205', '20250206', '20250207', '20250208', '20250209', '20250210', '20250211', '20250212', '20250213', '20250214', '20250215', '20250216', '20250217', '20250218', '20250219', '20250220', '20250221', '20250222', '20250223', '20250224', '20250225', '20250226', '20250227', '20250228', '20250301', '20250302', '2025

In [6]:
def get_data():
    for date in date_list:
    
        query_start_time = time.time()
        base_query = f"""
        
        select
            yyyymmdd,
            customer_id,
            service_obj_service_name service_name,
            order_id,
            order_status,
            pickup_cluster,
            drop_cluster,
            pickup_location_hex_8,
            drop_location_hex_8,
            pickup_location_hex_10,
            drop_location_hex_10,
            channel_host
        from 
            orders.order_logs_snapshot
        where 
            yyyymmdd = '{date}'
            and city_name = '{city}'
            and service_obj_service_name in ('Auto', 'Link', 'CabEconomy', 'Bike Lite', 'CabPremium', 'Cab SUV', 'Auto Pet', 'Auto C2C', 'Auto NCR','Auto Pool')
        """
        
        df_base_query = pd.read_sql(base_query, connection)
        query_end_time = time.time()
        
        extract_start_time = time.time()
        df_base_query.to_parquet(local_dataset + '/ban_gc_data_dump_{}.parquet'.format(date), index=False)
        extract_end_time = time.time()
        
        query_execution_time = query_end_time - query_start_time
        local_extract_time = extract_end_time - extract_start_time
        
        print(f"Query execution time: {query_execution_time:.6f} seconds & local extract time: {local_extract_time:.6f} seconds ")


# get_data()

In [7]:
def get_customer():
    
    parquet_files = glob.glob(os.path.join(local_dataset + "/*.parquet"))
    df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)    
    return df

df_data = get_customer()

In [8]:
print(df_data.shape)
df_data.head(5)

(62804568, 12)


,yyyymmdd,customer_id,service_name,order_id,order_status,pickup_cluster,drop_cluster,pickup_location_hex_8,drop_location_hex_8,pickup_location_hex_10,drop_location_hex_10,channel_host
0,20241216,65672495d2d97e2186886575,Link,675f927f21c53643714f5fed,customerCancelled,Basavanagudi,Metro Rashtreeya Vidyalaya,8861892597fffff,88618924a5fffff,8a618925965ffff,8a618924a487fff,android
1,20241216,6356453357f9a0741dd7cce3,Link,675fa174446da7409901f1c4,customerCancelled,Bommasandra Industrial Area,Jigani,8861893597fffff,8861892659fffff,8a6189359787fff,8a61892659a7fff,android
2,20241216,66532697b2087a0aafcb76e3,Auto Pool,67601d3904d7e84c46ee6ecd,dropped,AECS Layout Brookefield,Brookefield,88618920a3fffff,8861892e49fffff,8a618920a287fff,8a61892e48d7fff,android
3,20241216,623b28e49e4b441aa3c44cf5,Auto,6760333b10635f5fd6c2248a,dropped,Baiyappanahalli - CV Raman Nagar,BLR_Indiranagar,8861892ec5fffff,8861892ec3fffff,8a61892e124ffff,8a61892ec28ffff,android
4,20241216,6360f7deb207e1a8a083abcb,Auto,675fab8242e1910aaa9fef74,customerCancelled,Nagavara,Ramaiah College,8861892c83fffff,8860145941fffff,8a61892c82dffff,8a6014594197fff,android


In [9]:
df_data\
.groupby(['channel_host'])\
.agg(customers = ('customer_id', 'nunique'), orders = ('order_id', 'nunique'))\
.reset_index()

,channel_host,customers,orders
0,android,3690329,46717646
1,ios,871718,16086922


In [10]:
# Excluding Tummoc channel_host 
df_data = df_data[df_data['channel_host'].isin(['android','ios'])]

In [11]:
df_data\
.groupby(['channel_host'])\
.agg(customers = ('customer_id', 'nunique'), orders = ('order_id', 'nunique'))\
.reset_index()

,channel_host,customers,orders
0,android,3690329,46717646
1,ios,871718,16086922


### What is the total unique gross customer base over the past 90 days?

In [12]:
df_data.customer_id.nunique()

4529487

In [13]:
df_temp = df_data\
            .groupby(['customer_id'])\
            .agg(orders = ('order_id', 'nunique'))\
            .reset_index()

df_temp1 = df_temp\
            .groupby(['orders'])\
            .agg(customers = ('customer_id', 'nunique'))\
            .reset_index()\
            .sort_values(by='orders')

print(df_temp1.customers.sum())
print(df_data.customer_id.nunique())

4529487
4529487


In [14]:
df_temp1['cum_customers'] = df_temp1['customers'].cumsum()
df_temp1['total_customers'] = df_temp1.customers.sum()
df_temp1['% cum_customers'] = (df_temp1['cum_customers']*100.00/df_temp1['total_customers']).round(2)
df_temp1['% customers'] = (df_temp1['customers']*100.00/df_temp1['total_customers']).round(2)
df_temp1.head(15)

,orders,customers,cum_customers,total_customers,% cum_customers,% customers
0,1,813394,813394,4529487,17.96,17.96
1,2,563686,1377080,4529487,30.40,12.44
2,3,406256,1783336,4529487,39.37,8.97
3,4,314238,2097574,4529487,46.31,6.94
4,5,251697,2349271,4529487,51.87,5.56
5,6,207166,2556437,4529487,56.44,4.57
6,7,173773,2730210,4529487,60.28,3.84
7,8,147599,2877809,4529487,63.53,3.26
8,9,126796,3004605,4529487,66.33,2.80
9,10,111206,3115811,4529487,68.79,2.46


In [15]:
df_temp1[df_temp1['orders'] == 9][['orders', '% cum_customers']]

,orders,% cum_customers
8,9,66.33


### Applying geo-consistency logic on hex 8

In [16]:
list_customer_w_10p_rides = df_temp[df_temp['orders'] >= 10]['customer_id'].tolist()
len(list_customer_w_10p_rides)

1524882

In [17]:
df = df_data[df_data['customer_id'].isin(list_customer_w_10p_rides)][[ #'yyyymmdd', 
                                                                      'customer_id', 'order_id',
                                                                      # 'service_name', 'order_status', 
                                                                      # 'pickup_cluster', 'drop_cluster', 
                                                                      'pickup_location_hex_8', 'drop_location_hex_8', 
                                                                      # 'pickup_location_hex_10', 'drop_location_hex_10'
                                                                    ]]
df.columns

Index(['customer_id', 'order_id', 'pickup_location_hex_8',
       'drop_location_hex_8'],
      dtype='object')

In [18]:
df.head(5)

,customer_id,order_id,pickup_location_hex_8,drop_location_hex_8
0,65672495d2d97e2186886575,675f927f21c53643714f5fed,8861892597fffff,88618924a5fffff
3,623b28e49e4b441aa3c44cf5,6760333b10635f5fd6c2248a,8861892ec5fffff,8861892ec3fffff
6,6274d6f708aff062747ee8de,67605786107ff93b74feefc3,8860145b67fffff,8861892ccbfffff
7,6274d6f708aff062747ee8de,676038c527589c664a65b603,8861892ccbfffff,8860145b67fffff
9,606ae897b268460bc9dcde42,675fb6effd52b366d6467cb5,88618925c5fffff,88618925c7fffff


In [19]:
%%time

# Use frozenset to normalize routes (vectorized approach)
df["hex_8_route"] = [frozenset(x) for x in zip(df["pickup_location_hex_8"], df["drop_location_hex_8"])]

CPU times: user 2min 12s, sys: 3min 34s, total: 5min 46s
Wall time: 7min 29s


In [20]:
df.head(2)

,customer_id,order_id,pickup_location_hex_8,drop_location_hex_8,hex_8_route
0,65672495d2d97e2186886575,675f927f21c53643714f5fed,8861892597fffff,88618924a5fffff,"(8861892597fffff, 88618924a5fffff)"
3,623b28e49e4b441aa3c44cf5,6760333b10635f5fd6c2248a,8861892ec5fffff,8861892ec3fffff,"(8861892ec5fffff, 8861892ec3fffff)"


In [21]:
%%time

# Count occurrences of each route per customer
df_route_counts = df.groupby(["customer_id", "hex_8_route"]).size().reset_index(name="count")

# Total orders per customer
total_orders_per_customer = df["customer_id"].value_counts().reset_index()
total_orders_per_customer.columns = ["customer_id", "total_orders"]

CPU times: user 1min 24s, sys: 55.3 s, total: 2min 19s
Wall time: 2min 49s


In [22]:
%%time

# Merge and compute percentage per customer
df_route_counts = df_route_counts.merge(total_orders_per_customer, on="customer_id")
df_route_counts["percentage"] = (df_route_counts["count"] *100.00/ df_route_counts["total_orders"]).round()

CPU times: user 10.2 s, sys: 2.31 s, total: 12.6 s
Wall time: 12.6 s


In [23]:
df_route_counts.head(2)

,customer_id,hex_8_route,count,total_orders,percentage
0,5737c6adddbec2203f73316a,"(88618924abfffff, 8860145a15fffff)",1,10,10.0
1,5737c6adddbec2203f73316a,"(88618924abfffff, 8860145ad5fffff)",2,10,20.0


In [24]:
df_route_counts[df_route_counts['customer_id'] == '5737c6adddbec2203f73316a']

,customer_id,hex_8_route,count,total_orders,percentage
0,5737c6adddbec2203f73316a,"(88618924abfffff, 8860145a15fffff)",1,10,10.0
1,5737c6adddbec2203f73316a,"(88618924abfffff, 8860145ad5fffff)",2,10,20.0
2,5737c6adddbec2203f73316a,"(88618924abfffff, 8861892e93fffff)",2,10,20.0
3,5737c6adddbec2203f73316a,"(88618924abfffff, 8860145a2bfffff)",5,10,50.0


In [25]:
df_route_max_perc = df_route_counts.groupby(['customer_id']).agg(max_rr_route_perc = ('percentage','max')).reset_index()
df_route_max_perc

,customer_id,max_rr_route_perc
0,5737c6adddbec2203f73316a,50.0
1,5737c6aeddbec2203f733176,10.0
2,5737c6b1ddbec2203f73318b,30.0
3,5737c6c9ddbec2203f73325d,25.0
4,5737c6e0ddbec2203f73332c,38.0
...,...,...
1524877,67d565dc9ab13cfe1f6b4d69,90.0
1524878,67d56a7e46306f396b04c858,58.0
1524879,67d574759ab13c854571421a,27.0
1524880,67d590ea46306f11a31485cf,73.0


In [26]:
df_route_max_perc.shape

(1524882, 2)

In [27]:
df_route_max_perc.customer_id.nunique()

1524882

In [28]:
df_route_max_perc['max_rr_route_perc'].quantile([0.1,0.2,0.3,0.4,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95])

0.10    17.0
0.20    20.0
0.30    25.0
0.40    29.0
0.50    33.0
0.55    36.0
0.60    40.0
0.65    43.0
0.70    47.0
0.75    50.0
0.80    57.0
0.85    64.0
0.90    71.0
0.95    82.0
Name: max_rr_route_perc, dtype: float64

In [30]:
df_route_max_perc[df_route_max_perc['max_rr_route_perc'] >= 50.0].customer_id

,customer_id,max_rr_route_perc
0,5737c6adddbec2203f73316a,50.0
21,5737ca87ddbec220de03ebda,50.0
23,5737ca8eddbec220de03ec1b,57.0
28,5737d11addbec222694efcc4,65.0
31,5737d121ddbec222694efcf9,58.0
...,...,...
1524875,67d55953e70f9f63c1e70823,90.0
1524876,67d560c89ab13c456e696085,94.0
1524877,67d565dc9ab13cfe1f6b4d69,90.0
1524878,67d56a7e46306f396b04c858,58.0


In [43]:
# df_route_max_perc[['customer_id']].to_csv('gc_user_v1.csv', header=False,index=False)

'/Users/E2074/local-datasets/customer/geo-consistency'